<a href="https://colab.research.google.com/github/Esubaalew/8-slider-puzzle-problem/blob/main/8_slider_puzzle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 8-Puzzle Solver using A* Search

This notebook implements an A* search algorithm to solve the 8-slider (8-puzzle) problem. The solution uses the Manhattan Distance heuristic to efficiently guide the search. The notebook includes helper functions, the Node class definition, the A* search routine, and a demonstration with an example puzzle.


## Import Libraries

We import the necessary modules. We use `heapq` for the priority queue and Python’s typing for clearer function annotations.


In [35]:
import heapq
from typing import List, Tuple, Optional, Set




## Helper Functions: Manhattan Distance and Solvability Check

The `manhattan_distance` function computes the heuristic value for a given state relative to the goal state.
The `is_solvable` function checks if a given puzzle configuration is solvable by counting inversions.


In [36]:
def manhattan_distance(state: Tuple[int], goal: Tuple[int]) -> int:
    """Calculate the Manhattan distance of a given state compared to the goal state.

    Both state and goal are represented as tuples of 9 integers (0 is the blank).
    """
    distance = 0
    for i in range(9):
        if state[i] == 0:
            continue
        current_row, current_col = divmod(i, 3)
        goal_index = goal.index(state[i])
        goal_row, goal_col = divmod(goal_index, 3)
        distance += abs(current_row - goal_row) + abs(current_col - goal_col)
    return distance

def is_solvable(initial: Tuple[int], grid_width: int = 3) -> bool:
    """Check if the given 3x3 puzzle is solvable.

    For odd-width grids (like 3x3), the puzzle is solvable if the inversion count is even.
    """
    inv_count = 0
    initial_list = [num for num in initial if num != 0]
    for i in range(len(initial_list)):
        for j in range(i + 1, len(initial_list)):
            if initial_list[i] > initial_list[j]:
                inv_count += 1
    if grid_width % 2 != 0:  # Odd grid (3x3)
        return inv_count % 2 == 0
    else:
        # Even grid rules
        blank_row = divmod(initial.index(0), grid_width)[0]
        return (inv_count + blank_row) % 2 == 0


## Node Class and Successor Generation

The `Node` class holds the current state, a pointer to its parent, the move that led to it, and its cost metrics.
The `get_successors` function generates all valid moves from a given node by sliding the blank tile.


In [37]:
class Node:
    def __init__(self, state: Tuple[int], parent: Optional['Node'] = None, move: Optional[str] = None, g: int = 0, h: int = 0):
        self.state = state         # The puzzle state as a tuple.
        self.parent = parent       # Pointer to the parent node.
        self.move = move           # The move taken to reach this state.
        self.g = g                 # Cost to reach this node.
        self.h = h                 # Heuristic estimate from this node to the goal.
        self.f = g + h             # Total estimated cost (f = g + h).

    def __lt__(self, other: 'Node'):
        # Nodes are compared based on their f value for priority queue ordering.
        return self.f < other.f

def get_successors(node: Node, goal: Tuple[int]) -> List[Node]:
    """Generate successor nodes for a given node."""
    successors = []
    state = node.state
    blank_index = state.index(0)
    row, col = divmod(blank_index, 3)

    # Define possible moves with their row and column adjustments.
    moves = {
        'U': (-1, 0),
        'D': (1, 0),
        'L': (0, -1),
        'R': (0, 1)
    }

    for move, (dr, dc) in moves.items():
        new_row, new_col = row + dr, col + dc
        if 0 <= new_row < 3 and 0 <= new_col < 3:
            new_blank_index = new_row * 3 + new_col
            new_state = list(state)
            # Swap the blank with the target tile.
            new_state[blank_index], new_state[new_blank_index] = new_state[new_blank_index], new_state[blank_index]
            new_state_tuple = tuple(new_state)
            h = manhattan_distance(new_state_tuple, goal)
            successor = Node(new_state_tuple, parent=node, move=move, g=node.g + 1, h=h)
            successors.append(successor)
    return successors


## A* Search Implementation

The `a_star_search` function performs the A* search starting from the initial state until the goal state is reached.  
If a solution is found, we can reconstruct the move sequence using the `reconstruct_path` function.


In [38]:
def a_star_search(initial: Tuple[int], goal: Tuple[int]) -> Optional[Node]:
    if initial == goal:
        return Node(initial, None, None, 0, 0)

    # Call is_solvable with only the initial state (for a standard 3x3 puzzle)
    if not is_solvable(initial):
        print("The given puzzle configuration is unsolvable.")
        return None

    open_list = []
    closed_set: Set[Tuple[int]] = set()

    h = manhattan_distance(initial, goal)
    start_node = Node(initial, None, None, 0, h)
    heapq.heappush(open_list, start_node)

    while open_list:
        current_node = heapq.heappop(open_list)

        if current_node.state == goal:
            return current_node

        closed_set.add(current_node.state)

        for successor in get_successors(current_node, goal):
            if successor.state in closed_set:
                continue
            heapq.heappush(open_list, successor)

    return None



def reconstruct_path(node: Node) -> List[str]:
    """Reconstruct the sequence of moves from the initial state to the goal state."""
    moves = []
    while node.parent is not None:
        moves.append(node.move)
        node = node.parent
    moves.reverse()
    return moves



## Running the A* Search on an Example Puzzle

We define an example initial state and a goal state. Then, we run the A* search to find the solution path and print the results.


In [39]:

initial_state = (1, 2, 3,
                 4, 0, 6,
                 7, 5, 8)

goal_state = (1, 2, 3,
              4, 5, 6,
              7, 8, 0)


result = a_star_search(initial_state, goal_state)

if result:
    moves = reconstruct_path(result)
    print("Solution found!")
    print("Number of moves:", len(moves))
    print("Move sequence:", moves)
else:
    print("No solution exists for the given puzzle configuration.")


Solution found!
Number of moves: 2
Move sequence: ['D', 'R']


## Visualizing the Puzzle Solution

Below, we define a helper function to print the puzzle state in a 3x3 grid format. Then, we trace the path from the initial state to the goal state, printing each intermediate state.


In [40]:
def print_state(state: Tuple[int]):
    """Print the 3x3 state in a readable format."""
    for i in range(0, 9, 3):
        print(state[i:i+3])
    print()

if result:
    print("Initial State:")
    print_state(initial_state)


    node_path = []
    node = result
    while node is not None:
        node_path.append(node)
        node = node.parent
    node_path.reverse()

    for node in node_path:
        move_str = node.move if node.move is not None else "Start"
        print("Move:", move_str)
        print_state(node.state)


Initial State:
(1, 2, 3)
(4, 0, 6)
(7, 5, 8)

Move: Start
(1, 2, 3)
(4, 0, 6)
(7, 5, 8)

Move: D
(1, 2, 3)
(4, 5, 6)
(7, 0, 8)

Move: R
(1, 2, 3)
(4, 5, 6)
(7, 8, 0)

